# 无旧上限五会话事件：时钟与连续目标机械矩阵

事前冻结四类Qlib来源时钟与动作规则：普通早盘、半日市晚盘双向调仓、2020恢复日前/后。HOLD与EVENT从同一已结算账户状态出发，随后五会话停用旧b472与旧上限；检验时钟、整股Tiered、月度费用计数、隔夜/分钟可用回撤、终端股数现金净值。此卡无信息拟合或经济候选账户。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_capfree_lock_clock_matrix_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[24125:MainThread](2026-10-30 13:43:48,469) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[24125:MainThread](2026-10-30 13:43:48,471) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[24125:MainThread](2026-10-30 13:43:48,471) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[24125:MainThread](2026-10-30 13:43:48,543) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[24125:MainThread](2026-10-30 13:43:48,544) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[24125:MainThread](2026-10-30 13:43:48,544) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
by_date={};quotes_by_date={};read_count=0
for item in plan["cases"]:
 day=pd.Timestamp(item["trade_date"])
 if day in by_date:continue
 checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,
     recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],plan=parent_plan)
 by_date[day]=checkpoint
 last=frame.iloc[checkpoint.index+4]
 rawbars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],
     start_time=str(day.date())+" 09:31:00",end_time=str(pd.Timestamp(last.trade_date).date())+" 16:00:00",freq="1min")
 parts=[]
 for symbol in ("SOXL","SOXX"):
  x=rawbars.xs(symbol,level="instrument").copy();x.columns=[symbol+"_"+v.removeprefix("$") for v in x.columns];parts.append(x)
 quotes_by_date[day]=pd.concat(parts,axis=1).sort_index()
 read_count+=1
assert read_count==plan["budget"]["qlib_minute_reads"]
print('Qlib source reads',read_count)

Qlib source reads 3


In [3]:
results=[]
for item in plan["cases"]:
 day=pd.Timestamp(item["trade_date"]);checkpoint=by_date[day];quotes=quotes_by_date[day]
 bar_end=pd.Timestamp(str(day.date())+' '+item["bar_end_at"]+':00')
 fill=bar_end+pd.Timedelta(minutes=2);decision=bar_end+pd.Timedelta(minutes=1)
 late=bar_end>day+pd.Timedelta(hours=10,minutes=32)
 pos=book["native_positions"].get(day+pd.Timedelta(hours=10,minutes=32))
 assert pos is not None
 recovery_orders=book["orders"].loc[(book["orders"].trade_date==day)&(book["orders"].slot=='recovery')].to_dict('records')
 recovery_state=None
 if late:
  recovery_state={"quantities":np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ('SOXL','SOXX')],float),"cash":float(pos.get_cash()),"orders":recovery_orders}
 mark=quotes.loc[bar_end];mark_price=np.array([mark.SOXL_close,mark.SOXX_close],float)
 q=recovery_state["quantities"] if late else checkpoint.state.q
 cash=recovery_state["cash"] if late else checkpoint.state.cash
 nav=float(q@mark_price+cash);w=np.r_[q*mark_price,cash]/nav
 if item['target_rule']=='cash_to_soxx_0.10':target=w+np.array([0.,.10,-.10])
 elif item['target_rule']=='soxl_minus_0.04_soxx_plus_0.08_cash_minus_0.04':target=w+np.array([-.04,.08,-.04])
 else:raise ValueError('blocked: target rule absent')
 fillbar=quotes.loc[fill+pd.Timedelta(minutes=1)];adjusted=np.array([fillbar.SOXL_open,fillbar.SOXX_open],float);raw_fill=np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)
 event={'trade_date':day,'bar_end_at':bar_end,'decision_at':decision,'fill_at':fill,'desired_weights':target,'adjusted_fill_prices':adjusted,'raw_fill_prices':raw_fill}
 out=replay_capfree_lock_episode(frame,checkpoint,quotes,event=event,recovery_state=recovery_state,horizon_sessions=5)
 hold=out['HOLD'];branch=out['EVENT'];terminal=frame.iloc[checkpoint.index+4][['soxl_exit_open','soxx_exit_open']].to_numpy(float)
 delta=float(branch['path'].iloc[-1].next_open_nav-hold['path'].iloc[-1].next_open_nav)
 direct=float((branch['ending_quantities']-hold['ending_quantities'])@terminal+branch['ending_cash']-hold['ending_cash'])
 assert abs(delta-direct)<1e-7 and branch['ending_cash']>=-1e-8
 assert branch['path'].event_fee_usd.iloc[1:].eq(0).all() and hold['path'].event_fee_usd.eq(0).all()
 expected_extra=sum(int(o['raw_shares']) for o in branch['orders'] if not o['cap_applied'])
 month=fill.strftime('%Y-%m');actual_extra=branch['monthly_fee_shares'].get(month,0)-hold['monthly_fee_shares'].get(month,0)
 assert actual_extra==expected_extra
 parent_next=float(book['independent_replay'].iloc[checkpoint.index].native_expected_nav)
 hold_parent_error=abs(float(hold['path'].iloc[0].next_open_nav)-parent_next)
 if not book['independent_replay'].iloc[checkpoint.index].recovery_action or late:assert hold_parent_error<1e-7
 if late:
  assert out['post_1032_parent_state_spliced']
  assert np.max(np.abs(out['prefill_quantities']-recovery_state['quantities']))==0
  assert abs(out['prefill_cash']-recovery_state['cash'])==0
 else:assert not out['post_1032_parent_state_spliced']
 results.append({'case':item['name'],'day':str(day.date()),'clock':str(bar_end)+'->'+str(fill),'session_close':str(out['session_close']),'recovery_orders_before_decision':len(recovery_orders) if late else 0,'post_1032_parent_state_spliced':out['post_1032_parent_state_spliced'],'event_order_count':len(branch['orders']),'event_order_symbols':[o['symbol'] for o in branch['orders']],'event_order_sell':[bool(o['sell']) for o in branch['orders']],'event_fee_usd':float(sum(o['total'] for o in branch['orders'])),'monthly_fee_share_delta':actual_extra,'event_status':out['event_status'],'hold_parent_first_next_open_error_usd':hold_parent_error,'terminal_nav_closure_error_usd':abs(delta-direct),'hold_available_mark_max_drawdown':float(hold['path'].available_mark_max_drawdown.max()),'event_available_mark_max_drawdown':float(branch['path'].available_mark_max_drawdown.max())})
# The three adjacent bars have unresolved recovery/decision ordering and must fail closed.
ambiguous=[]
checkpoint=by_date[pd.Timestamp('2020-03-09')];quotes=quotes_by_date[pd.Timestamp('2020-03-09')]
for hhmm in plan['ambiguous_recovery_bar_ends']:
 day=pd.Timestamp('2020-03-09');bar_end=pd.Timestamp(str(day.date())+' '+hhmm+':00');fill=bar_end+pd.Timedelta(minutes=2)
 fillbar=quotes.loc[fill+pd.Timedelta(minutes=1)]
 event={'trade_date':day,'bar_end_at':bar_end,'decision_at':bar_end+pd.Timedelta(minutes=1),'fill_at':fill,'desired_weights':np.array([0.,1.,0.]),'adjusted_fill_prices':np.array([fillbar.SOXL_open,fillbar.SOXX_open],float),'raw_fill_prices':np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)}
 try:replay_capfree_lock_episode(frame,checkpoint,quotes,event=event)
 except ValueError as exc:assert 'ambiguous' in str(exc);ambiguous.append(hhmm)
 else:raise AssertionError('ambiguous clock accepted')
assert len(ambiguous)==3
# Synthetic zero-order target checks no-fill is retained as an episode, not dropped.
case=plan['cases'][0];day=pd.Timestamp(case['trade_date']);checkpoint=by_date[day];quotes=quotes_by_date[day]
bar_end=pd.Timestamp(str(day.date())+' '+case['bar_end_at']+':00');fill=bar_end+pd.Timedelta(minutes=2);fillbar=quotes.loc[fill+pd.Timedelta(minutes=1)]
adjusted=np.array([fillbar.SOXL_open,fillbar.SOXX_open],float);raw_fill=np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)
nav=float(checkpoint.state.q@adjusted+checkpoint.state.cash);flat=np.r_[checkpoint.state.q*adjusted,checkpoint.state.cash]/nav
synthetic={'trade_date':day,'bar_end_at':bar_end,'decision_at':bar_end+pd.Timedelta(minutes=1),'fill_at':fill,'desired_weights':flat,'adjusted_fill_prices':adjusted,'raw_fill_prices':raw_fill}
zero=replay_capfree_lock_episode(frame,checkpoint,quotes,event=synthetic)
assert zero['event_status']=='no_whole_share_fill' and not zero['EVENT']['orders']
assert zero['EVENT']['path'].next_open_nav.equals(zero['HOLD']['path'].next_open_nav)
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'module_sha256':sha(root/plan['module']),'cases':results,'ambiguous_recovery_bars_rejected':ambiguous,'synthetic_no_fill_retained':True,'qlib_minute_reads':read_count,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'four_case_capfree_lock_mechanical_clock_matrix_pass_not_bulk_labels'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_capfree_lock_clock_matrix_20261030",
  "plan_sha256": "bc4e8845eb6291d98684927ae4391781f38bbe76ee3fc9456be5592dcbeeca77",
  "module_sha256": "1b3d8988db90a29ddc5700759cc4aebabfc32ddd3d126fc84188fa6a1d1fd3c1",
  "cases": [
    {
      "case": "ordinary_early",
      "day": "2019-12-19",
      "clock": "2019-12-19 09:48:00->2019-12-19 09:50:00",
      "session_close": "2019-12-19 16:00:00",
      "recovery_orders_before_decision": 0,
      "post_1032_parent_state_spliced": false,
      "event_order_count": 1,
      "event_order_symbols": [
        "SOXX"
      ],
      "event_order_sell": [
        false
      ],
      "event_fee_usd": 0.55524925,
      "monthly_fee_share_delta": 64.0,
      "event_status": "funded",
      "hold_parent_first_next_open_error_usd": 0.0,
      "terminal_nav_closure_error_usd": 2.1827872842550278e-11,
      "hold_available_mark_max_drawdown": 0.009570343031858108,
      "event_available_mark_max_drawdown": 0.010218654298004726
  

## 中文结论

**SOXX：无新候选账户、无0/2/5bp通过。** 四个事前固定来源案例均以Qlib分钟/原生父本、同起点HOLD与EVENT、整股Tiered独立费用和五会话锁定运行。普通早盘SOXX单买、2018-11-23半日市13:00收盘前SOXL卖出＋SOXX买入、2020-03-09恢复前后均通过终端自身股数/现金NAV恒等式，最大误差2.18e−11美元；费用月度股数计数增量逐单闭合。恢复日前早盘锁定的HOLD比父本首日次开盘差19.4067美元，原因是两支都取消了原本10:32的恢复动作；恢复后12:00先接入一笔已结算父本恢复订单，HOLD对父本首日误差5.82e−11美元。半日市12:00仅接入原生10:32持仓，恢复订单为零。

10:30/10:31/10:32三个靠近恢复时刻的完成bar因顺序不明被拒绝；合成零整股订单仍保留`no_whole_share_fill`，EVENT与HOLD路径相同。五会话可用mark回撤包括次开盘，不认证异步分钟OHLC真峰谷。日期、时刻按来源和恢复覆盖预选，非收益挑样；2021/2022目标值未参加。此接口可处理单事件的早/晚、半日市及连续三资产simplex目标，但尚未批量扫描全部合格分钟、未处理重复触发锁定队列或模型连续输出。下一步先对全部来源分钟做时钟/状态/整股支持、成熟期和计算预算门，之后才准入路径标签/信息拟合。